In [ ]:
# Euro FX Reference Rates — Exploratory Analysis

Data flows from the [ECB reference rate CSV](https://csvbase.com/table-munger/eurofxref-hist)
into DuckDB via `extract.py`, transformed by dbt, and queried here.

**Layers used**
| Table | Description |
|---|---|
| `raw.fx_rates` | Raw long-format source data |
| `main.int_fx_rates_cleaned` | Cleaned rates with `eur_per_unit` |
| `main.mart_fx_latest` | Most recent rate per currency |

In [ ]:
import os
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Connect to the local DuckDB database (repo root)
DB_PATH = os.environ.get("DUCKDB_PATH", "../duckdb.db")
con = duckdb.connect(DB_PATH, read_only=True)
print(f"Connected to {DB_PATH}")
print("Available tables:")
con.execute("SHOW ALL TABLES").df()

## 1 — Raw data overview

In [ ]:
raw = con.execute("""
    SELECT
        COUNT(*)                                        AS total_rows,
        COUNT(DISTINCT currency)                        AS currencies,
        MIN(date::date)                                 AS earliest,
        MAX(date::date)                                 AS latest,
        COUNT(*) FILTER (WHERE rate IS NULL)            AS null_rates,
        ROUND(AVG(rate) FILTER (WHERE rate IS NOT NULL), 4) AS avg_rate
    FROM raw.fx_rates
""").df()
raw

## 2 — Latest rates (mart)

In [ ]:
latest = con.execute("""
    SELECT currency, rate, eur_per_unit, latest_date
    FROM main.mart_fx_latest
    ORDER BY rate DESC
""").df()

print(f"{len(latest)} currencies  —  latest date: {latest['latest_date'].max()}")
latest.head(10)

## 3 — Chart: strongest and weakest currencies vs EUR

In [ ]:
# Top 10 (most units per EUR) and bottom 10 (fewest units per EUR)
top10    = latest.nlargest(10,  "rate")
bottom10 = latest.nsmallest(10, "rate")
chart_df = pd.concat([top10, bottom10]).sort_values("rate", ascending=True)

fig = px.bar(
    chart_df,
    x="rate",
    y="currency",
    orientation="h",
    color="rate",
    color_continuous_scale="RdYlGn_r",
    title=f"10 weakest and 10 strongest currencies vs EUR  ({latest['latest_date'].max()})",
    labels={"rate": "Units per 1 EUR", "currency": ""},
)
fig.update_layout(coloraxis_showscale=False, height=550)
fig.show()

## 4 — Time series: rate history for selected currencies

In [ ]:
CURRENCIES = ["USD", "GBP", "JPY", "CHF", "CNY"]

history = con.execute(f"""
    SELECT fx_date, currency, rate
    FROM main.int_fx_rates_cleaned
    WHERE currency IN ({','.join(f"'{c}'" for c in CURRENCIES)})
    ORDER BY fx_date
""").df()

fig = px.line(
    history,
    x="fx_date",
    y="rate",
    color="currency",
    title="ECB reference rate history (1 EUR = N units)",
    labels={"fx_date": "Date", "rate": "Units per 1 EUR", "currency": ""},
)
fig.update_layout(height=450, hovermode="x unified")
fig.show()

## 5 — Year-on-year average rate per currency

In [ ]:
yearly = con.execute(f"""
    SELECT
        EXTRACT('year' FROM fx_date)::INT AS year,
        currency,
        ROUND(AVG(rate), 4) AS avg_rate
    FROM main.int_fx_rates_cleaned
    WHERE currency IN ({','.join(f"'{c}'" for c in CURRENCIES)})
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

fig = px.line(
    yearly,
    x="year",
    y="avg_rate",
    color="currency",
    markers=True,
    title=f"Annual average rate vs EUR — {', '.join(CURRENCIES)}",
    labels={"year": "Year", "avg_rate": "Annual avg (units per EUR)", "currency": ""},
)
fig.update_layout(height=420, hovermode="x unified")
fig.show()

## 6 — Volatility: annual rate standard deviation (top 20 currencies)

In [ ]:
vol = con.execute("""
    SELECT
        currency,
        ROUND(STDDEV(rate), 4)              AS rate_std,
        ROUND(AVG(rate), 4)                 AS rate_mean,
        ROUND(STDDEV(rate) / AVG(rate), 4)  AS coeff_of_variation
    FROM main.int_fx_rates_cleaned
    GROUP BY currency
    HAVING COUNT(*) > 100          -- enough data points
    ORDER BY coeff_of_variation DESC
    LIMIT 20
""").df()

fig = px.bar(
    vol.sort_values("coeff_of_variation"),
    x="coeff_of_variation",
    y="currency",
    orientation="h",
    color="coeff_of_variation",
    color_continuous_scale="Reds",
    title="Top 20 most volatile currencies (coefficient of variation since 1999)",
    labels={"coeff_of_variation": "StdDev / Mean", "currency": ""},
)
fig.update_layout(coloraxis_showscale=False, height=540)
fig.show()

## 7 — Correlation heatmap (selected currencies, 2015–present)

In [ ]:
CORR_CURRENCIES = ["USD", "GBP", "JPY", "CHF", "CNY", "SEK", "NOK", "DKK", "CAD", "AUD"]

pivot = con.execute(f"""
    PIVOT (
        SELECT fx_date, currency, rate
        FROM main.int_fx_rates_cleaned
        WHERE currency IN ({','.join(f"'{c}'" for c in CORR_CURRENCIES)})
          AND fx_date >= '2015-01-01'
    )
    ON currency
    USING AVG(rate)
    GROUP BY fx_date
""").df().set_index("fx_date").dropna()

corr = pivot.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="Rate correlation matrix (2015–present)",
    aspect="auto",
)
fig.update_layout(height=480)
fig.show()

In [ ]:
con.close()
print("Done.")